# MoCo Colab Run

This notebook clones the latest `method/moco-v2` branch, copies `dataset.zip` from Google Drive to Colab local SSD, writes all `./output` artifacts into a Drive-backed run folder, and runs MoCo pretraining, fine-tuning, and evaluation.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUN_NAME = f'moco_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_ZIP =', DATASET_ZIP)
print('DRIVE_RUN_DIR =', DRIVE_RUN_DIR)

PROJECT_DIR = /content/SSL_Prostate_Cancer_Grading
DATASET_ZIP = /content/drive/MyDrive/Prostate_SSL/dataset.zip
DRIVE_RUN_DIR = /content/drive/MyDrive/Prostate_SSL/runs/moco_20260328_235610


In [5]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

/content
Cloning into '/content/SSL_Prostate_Cancer_Grading'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 211 (delta 89), reused 179 (delta 60), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 9.51 MiB | 17.45 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/SSL_Prostate_Cancer_Grading
method/moco-v2
06cd6c9 (HEAD -> method/moco-v2, origin/method/moco-v2) Succesfully ran on colab, added notes and plan


## Apply local patches

Since you don't have write access to the upstream repo, this cell patches `finetune_moco.py` to add CLI args for controlling fine-tune epochs and batch size. Skip this cell if the upstream repo already has these changes.


In [ ]:
%cd "$PROJECT_DIR"

# Patch finetune_moco.py to add --epochs_stage1, --epochs_stage2, --batch_size CLI args
import re

finetune_path = 'training/moco/finetune_moco.py'
with open(finetune_path, 'r') as f:
    content = f.read()

# Check if already patched
if '--epochs_stage1' in content:
    print('finetune_moco.py already has CLI args for epochs - skipping patch')
else:
    print('Patching finetune_moco.py to add epoch/batch CLI args...')
    
    # Patch 1: Add CLI args to parse_args()
    content = content.replace(
        '''def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune MoCo encoder for classification")
    parser.add_argument(
        "--checkpoint",
        type=str,
        default=None,
        help="Path to encoder_q .weights.h5 or full-state MoCo checkpoint",
    )
    return parser.parse_args()''',
        '''def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune MoCo encoder for classification")
    parser.add_argument(
        "--checkpoint",
        type=str,
        default=None,
        help="Path to encoder_q .weights.h5 or full-state MoCo checkpoint",
    )
    parser.add_argument(
        "--epochs_stage1",
        type=int,
        default=None,
        help="Epochs for Stage 1 (head-only). Overrides EPOCHS_STAGE_1 constant.",
    )
    parser.add_argument(
        "--epochs_stage2",
        type=int,
        default=None,
        help="Epochs for Stage 2 (partial unfreeze). Overrides EPOCHS_STAGE_2 constant.",
    )
    parser.add_argument(
        "--batch_size",
        type=int,
        default=None,
        help="Batch size for fine-tuning. Overrides BATCH_SIZE constant.",
    )
    return parser.parse_args()'''
    )
    
    # Patch 2: Wire args into main() before training starts
    content = content.replace(
        '''def main():
    args = parse_args()
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    checkpoint_path = args.checkpoint or latest_moco_encoder_weights()
    if not checkpoint_path:
        raise FileNotFoundError(
            "No MoCo checkpoint found. Run training/moco/pretrain_moco.py first or pass --checkpoint."
        )

    print("=" * 70)
    print("MoCo Fine-Tuning for Gleason Grading")
    print("=" * 70)
    print(f"Checkpoint: {checkpoint_path}")
    print_device_configuration()
    print("=" * 70)''',
        '''def main():
    args = parse_args()
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Allow CLI overrides of module-level constants
    epochs_stage1 = args.epochs_stage1 if args.epochs_stage1 is not None else EPOCHS_STAGE_1
    epochs_stage2 = args.epochs_stage2 if args.epochs_stage2 is not None else EPOCHS_STAGE_2
    batch_size = args.batch_size if args.batch_size is not None else BATCH_SIZE

    checkpoint_path = args.checkpoint or latest_moco_encoder_weights()
    if not checkpoint_path:
        raise FileNotFoundError(
            "No MoCo checkpoint found. Run training/moco/pretrain_moco.py first or pass --checkpoint."
        )

    print("=" * 70)
    print("MoCo Fine-Tuning for Gleason Grading")
    print("=" * 70)
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Stage 1 epochs: {epochs_stage1} | Stage 2 epochs: {epochs_stage2} | Batch size: {batch_size}")
    print_device_configuration()
    print("=" * 70)'''
    )
    
    # Patch 3: Replace BATCH_SIZE constants with batch_size variable
    content = re.sub(
        r'batch_size=BATCH_SIZE,(\s+path_to_img=IMG_DIR,)',
        r'batch_size=batch_size,\1',
        content
    )
    
    # Patch 4: Replace EPOCHS_STAGE_1/2 with local vars
    content = content.replace('epochs=EPOCHS_STAGE_1,', 'epochs=epochs_stage1,')
    content = content.replace('if EPOCHS_STAGE_2 > 0:', 'if epochs_stage2 > 0:')
    content = content.replace('epochs=EPOCHS_STAGE_2,', 'epochs=epochs_stage2,')
    
    # Patch 5: Fix summary dict and plot markers
    content = content.replace('"stage1_epochs": EPOCHS_STAGE_1,', '"stage1_epochs": epochs_stage1,')
    content = content.replace('"stage2_epochs": EPOCHS_STAGE_2,', '"stage2_epochs": epochs_stage2,')
    content = content.replace('plt.axvline(x=EPOCHS_STAGE_1,', 'plt.axvline(x=epochs_stage1,')
    
    with open(finetune_path, 'w') as f:
        f.write(content)
    
    print('✓ Patched finetune_moco.py successfully')


In [4]:
%cd /content/SSL_Prostate_Cancer_Grading

!python -m pip install --upgrade pip
!grep -Ev '^(tensorflow|keras|tensorboard|protobuf|h5py|ml_dtypes)(==|>|<)' requirements.txt > requirements-colab-lite.txt
!pip install -r requirements-colab-lite.txt


/content/SSL_Prostate_Cancer_Grading
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 116.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 141.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 94.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 170.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 155.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 179.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 60.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 233.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 93.7 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation

In [6]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))


2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
%cd "$PROJECT_DIR"
!test -f "$DATASET_ZIP" || (echo "Missing dataset zip at $DATASET_ZIP" && exit 1)
!rm -rf ./dataset
!cp "$DATASET_ZIP" ./dataset.zip
!unzip -q ./dataset.zip
!rm -f ./dataset.zip
!ls ./dataset | head

/content/SSL_Prostate_Cancer_Grading
images
masks
panda_images
PANDA-PLUS-Bench
partition
Pretrain_Manifest.csv
Test.csv
Train.csv
TrainSplit.csv
Val.csv


In [8]:
%cd "$PROJECT_DIR"
!mkdir -p "$DRIVE_RUN_DIR/output"
!mkdir -p "$DRIVE_RUN_DIR/notebook_logs"
!rm -rf ./output
!ln -s "$DRIVE_RUN_DIR/output" ./output
!echo "$RUN_NAME" > "$DRIVE_RUN_DIR/notebook_logs/run_name.txt"
!pwd
!ls -ld ./output
!ls -ld "$DRIVE_RUN_DIR/output"

/content/SSL_Prostate_Cancer_Grading
/content/SSL_Prostate_Cancer_Grading
lrwxrwxrwx 1 root root 68 Mar 28 23:59 ./output -> /content/drive/MyDrive/Prostate_SSL/runs/moco_20260328_235610/output
drwx------ 2 root root 4096 Mar 28 23:59 /content/drive/MyDrive/Prostate_SSL/runs/moco_20260328_235610/output


In [9]:
%cd "$PROJECT_DIR"
import os

required_files = [
    './dataset/Train.csv',
    './dataset/Test.csv',
    './dataset/TrainSplit.csv',
    './dataset/Val.csv',
    './dataset/Pretrain_Manifest.csv',
]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    print('Missing generated dataset files:', missing)
    !python data/setup.py
else:
    print('Dataset CSVs and manifest already present.')

/content/SSL_Prostate_Cancer_Grading
Dataset CSVs and manifest already present.


## Pretrain

Edit `EPOCHS`, `BATCH_SIZE`, and `SAVE_FREQ` before running. This cell writes checkpoints directly under the Drive-backed `./output` symlink, so no extra copy step is required.

In [ ]:
%cd "$PROJECT_DIR"
EPOCHS = 100        # teammate ran 50; increase to 100 for better representations
BATCH_SIZE = 128    # teammate ran 64; T4=128 safe, A100 can try 256
SAVE_FREQ = 10
RESUME = ''  # Example: './output/models/moco/state/moco_state-10'

# LR scales linearly with batch: base_lr * (batch_size / 256)
# Default script base_lr is 0.03 (tuned for batch 256).
# At batch 128 the effective LR is already halved by the schedule; leave LR unset
# unless you change BATCH_SIZE significantly (e.g. 256 → add: --lr 0.03)
cmd = f'python training/moco/pretrain_moco.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --save_freq {SAVE_FREQ}'
if RESUME.strip():
    cmd += f' --resume {RESUME}'
print(cmd)
!$cmd


/content/SSL_Prostate_Cancer_Grading
python training/moco/pretrain_moco.py --epochs 100 --batch_size 128 --save_freq 10
2026-03-29 00:00:19.427435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774742419.436859   23634 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774742419.439922   23634 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774742419.447588   23634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774742419.447602   23634 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking

## Fine-Tune

This cell picks the latest saved `encoder_q_epoch*.weights.h5` checkpoint unless you override it.

In [ ]:
%cd "$PROJECT_DIR"
import glob

# ── Fine-tune settings ────────────────────────────────────────────────────────
FINETUNE_BATCH_SIZE = 32    # teammate default was 8; T4 comfortably handles 32
EPOCHS_STAGE_1 = 50         # head-only (frozen encoder); was 30
EPOCHS_STAGE_2 = 20         # partial unfreeze; was 10
# ─────────────────────────────────────────────────────────────────────────────

checkpoint_candidates = sorted(glob.glob('./output/models/moco/encoder_q_epoch*.weights.h5'))
if not checkpoint_candidates:
    raise FileNotFoundError('No MoCo encoder checkpoints found under ./output/models/moco/')

CHECKPOINT = checkpoint_candidates[-1]
print('Using checkpoint:', CHECKPOINT)

cmd = (
    f'python training/moco/finetune_moco.py'
    f' --checkpoint "{CHECKPOINT}"'
    f' --epochs_stage1 {EPOCHS_STAGE_1}'
    f' --epochs_stage2 {EPOCHS_STAGE_2}'
    f' --batch_size {FINETUNE_BATCH_SIZE}'
)
print(cmd)
!$cmd


/content/SSL_Prostate_Cancer_Grading
Using checkpoint: ./output/models/moco/encoder_q_epoch050.weights.h5
I0000 00:00:1773984105.305771   62862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773984107.117759   62862 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
MoCo Fine-Tuning for Gleason Grading
Checkpoint: ./output/models/moco/encoder_q_epoch050.weights.h5
W0000 00:00:1773984107.723922   62862 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer

## Evaluate

This uses `best_moco_overall.keras` by default. All evaluation artifacts are written into the Drive-backed `./output/moco/` folder.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluation/moco/eval_moco.py

/content/SSL_Prostate_Cancer_Grading
I0000 00:00:1773984734.398131   76364 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773984736.203815   76364 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
MoCo Classifier Evaluation
Test samples: 2487
W0000 00:00:1773984736.817311   76364 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1773984736.824217   76364 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible w

## Sync Runtime State to Drive

Use this after pretraining, fine-tuning, or evaluation if you want an explicit sync of generated runtime artifacts beyond the Drive-backed `./output` folder. This copies generated dataset CSVs/manifests, the notebook itself, and a small run summary into the current Drive run directory.

In [ ]:
%cd "$PROJECT_DIR"
!mkdir -p "$DRIVE_RUN_DIR/runtime_sync/dataset"
!mkdir -p "$DRIVE_RUN_DIR/runtime_sync/notebook"
!cp -f ./run_colab.ipynb "$DRIVE_RUN_DIR/runtime_sync/notebook/"
!for f in Train.csv Test.csv TrainSplit.csv Val.csv Pretrain_Manifest.csv; do if [ -f "./dataset/$f" ]; then cp -f "./dataset/$f" "$DRIVE_RUN_DIR/runtime_sync/dataset/"; fi; done
!git rev-parse HEAD > "$DRIVE_RUN_DIR/runtime_sync/commit.txt"
!find ./output -maxdepth 4 -type f | sort > "$DRIVE_RUN_DIR/runtime_sync/output_manifest.txt"
!echo "Synced runtime artifacts to: $DRIVE_RUN_DIR/runtime_sync"

/content/SSL_Prostate_Cancer_Grading
cp: cannot create regular file '/runtime_sync/dataset/': No such file or directory
cp: cannot create regular file '/runtime_sync/dataset/': No such file or directory
cp: cannot create regular file '/runtime_sync/dataset/': No such file or directory
cp: cannot create regular file '/runtime_sync/dataset/': No such file or directory
cp: cannot create regular file '/runtime_sync/dataset/': No such file or directory
Synced runtime artifacts to: /content/drive/MyDrive/Prostate_SSL/runs/moco_20260320_023752/runtime_sync


In [ ]:
%cd "$PROJECT_DIR"

import os
import shutil
from pathlib import Path

sync_root = Path(DRIVE_RUN_DIR) / "runtime_sync"
dataset_sync = sync_root / "dataset"
notebook_sync = sync_root / "notebook"

dataset_sync.mkdir(parents=True, exist_ok=True)
notebook_sync.mkdir(parents=True, exist_ok=True)

shutil.copy2("run_colab.ipynb", notebook_sync / "run_colab.ipynb")

for name in ["Train.csv", "Test.csv", "TrainSplit.csv", "Val.csv", "Pretrain_Manifest.csv"]:
    src = Path("dataset") / name
    if src.exists():
        shutil.copy2(src, dataset_sync / name)

os.system(f'git rev-parse HEAD > "{sync_root / "commit.txt"}"')
os.system(f'find ./output -maxdepth 4 -type f | sort > "{sync_root / "output_manifest.txt"}"')

print(f"Synced runtime artifacts to: {sync_root}")


/content/SSL_Prostate_Cancer_Grading
Synced runtime artifacts to: /content/drive/MyDrive/Prostate_SSL/runs/moco_20260320_023752/runtime_sync


In [ ]:
%cd "$PROJECT_DIR"
!echo "Run directory: $DRIVE_RUN_DIR"
!find ./output -maxdepth 3 -type f | sort | tail -n 40

/content/SSL_Prostate_Cancer_Grading
Run directory: /content/drive/MyDrive/Prostate_SSL/runs/moco_20260320_023752
